# ComfyUI on Google Colab v5e-1 TPU
Select **Runtime → Change runtime type → TPU (v5e-1)** before running. Cache import/export is local and portable; Google Drive is optional only for models.


In [ ]:
import os, platform, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/kevinmetten/ComfyUI-TPU.git'
# Use 'master' after PR #1 is merged. This value selects the existing PR branch for validation.
BRANCH = 'codex/build-comfyui-backend-for-google-colab-tpu'
# Leave unset for requirements.txt's upstream comfy-kitchen. Set a Git URL only after TPU profiling justifies the fork.
COMFY_KITCHEN_SPEC = None
ROOT = Path('/content/ComfyUI-TPU')
CACHE_DIR = ROOT / '.cache' / 'tpu_xla'
print('Repository:', REPO_URL)
print('Branch:', BRANCH)
print('Python:', platform.python_version())
print('Cache directory:', CACHE_DIR)


## Clone and prepare the TPU runtime
The notebook first checks out the selected version-controlled setup helper. That helper queries the official PyTorch/XLA TPU wheel index, resolves exact compatible package versions, and skips installation when the current complete stack is already compatible.


In [ ]:
if not (ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(ROOT)])
else:
    subprocess.check_call(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(ROOT), 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir(ROOT)
from tools.tpu_colab_setup import detect_tpu
tpu_detection = detect_tpu()
assert tpu_detection['detected'], f'Select a TPU runtime; detection details: {tpu_detection}'
print('TPU detection:', tpu_detection)
subprocess.check_call([sys.executable, '-m', 'tools.tpu_colab_setup', '--ensure'])


In [ ]:
import importlib.metadata
import re
import tempfile
from tools.tpu_colab_setup import installed_stack, stack_is_compatible

selected_stack = installed_stack()
assert stack_is_compatible(selected_stack), f'Incompatible TPU stack after setup: {selected_stack}'
requirements = (ROOT / 'requirements.txt').read_text().splitlines()
requirements = [line for line in requirements if not re.match(r'^\s*(torch|torchvision|torchaudio)(?:\s|[<>=!~]|$)', line, re.I)]
with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as filtered:
    filtered.write('\n'.join(requirements) + '\n')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', filtered.name])
if COMFY_KITCHEN_SPEC:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', COMFY_KITCHEN_SPEC])
assert installed_stack() == selected_stack, f'ComfyUI dependencies changed the TPU stack: expected {selected_stack}, found {installed_stack()}'
import torch
import torch_xla
print('Torch:', torch.__version__)
print('Torch/XLA:', importlib.metadata.version('torch-xla'))
print('TorchVision:', importlib.metadata.version('torchvision'))
print('TorchAudio:', importlib.metadata.version('torchaudio'))
print('Comfy-Kitchen:', importlib.metadata.version('comfy-kitchen'))
print('ComfyUI commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Optional cache import
Run this before any TPU operation. Cancel the upload dialog to start with an empty cache.


In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        subprocess.check_call([sys.executable, '-m', 'tools.tpu_cache', 'import', name])
        break


## Capability probe
Review every operation. An `error` is an unsupported path, not a passed test.


In [ ]:
PROBE_REPORT = ROOT / 'tpu_probe.json'
DIAGNOSTICS = ROOT / 'diagnostics'
subprocess.run([sys.executable, '-m', 'tools.tpu_probe', '--output', str(PROBE_REPORT), '--diagnostics', str(DIAGNOSTICS), '--int8-experiments', '--large'], check=True)
print('TPU capability probe completed successfully.')
print('Report:', PROBE_REPORT)
print('Diagnostics:', DIAGNOSTICS)


## Optional model storage
Local `/content` paths work directly. Mount Drive only if desired, then configure its directories with `extra_model_paths.yaml`. Hugging Face downloads should be explicitly initiated by the user.


In [ ]:
# Optional:
# from google.colab import drive
# drive.mount('/content/drive')


## Launch and tunnel
This downloads Cloudflare's tunnel binary and starts ComfyUI. The printed `trycloudflare.com` URL is the remote UI.


In [ ]:
import urllib.request
from tools.tpu_colab_launch import run_colab_server

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
print('The cell remains active while ComfyUI and the tunnel are healthy. Stop the cell to shut them down.')
run_colab_server(ROOT, cloudflared)


## Export cache to your computer


In [ ]:
archive = ROOT / 'comfyui-v5e-tpu-cache.zip'
subprocess.check_call([sys.executable, '-m', 'tools.tpu_cache', 'export', str(archive)])
files.download(str(archive))
